# 🏥 TriMedAgent Full Stack - Multi-turn Chat + RAG + Full Pipeline

**Demo hoàn chỉnh của TriMedAgent với tất cả tính năng:**

| Feature | Description |
|---------|-------------|
| 🔄 **Multi-turn Memory** | Hội thoại nhiều lượt với context awareness |
| 📚 **Medical RAG** | Tra cứu kiến thức y khoa với Groq API |
| 🔬 **Visual Pipeline** | Triage → Reasoning → Detection → Segmentation |
| 💬 **Gradio Interface** | Chat UI tương tác đẹp |

---

## Architecture

```
User Input (Text + Image)
       │
       ▼
┌─────────────────────────────────────┐
│     TriMedOrchestratorV2            │
│  ┌────────────────────────────┐     │
│  │  ConversationState         │     │
│  │  ├─ messages (history)     │     │
│  │  ├─ images (per-image)     │     │
│  │  └─ summary (auto)         │     │
│  └────────────────────────────┘     │
│              │                      │
│  ┌───────────┼───────────┐         │
│  ▼           ▼           ▼         │
│ Triage    Reasoning    RAG         │
│ (CLIP)    (LLaVA)    (Groq)        │
│              │                      │
│  ┌───────────┴───────────┐         │
│  ▼                       ▼         │
│ Detection            Segmentation  │
│ (DINO+GK)              (MedSAM)    │
└─────────────────────────────────────┘
```

---
## 1️⃣ Environment Setup

In [ ]:
# Install required packages
!pip install -q groq gradio pillow requests numpy scipy sentence-transformers

In [ ]:
import os
import sys
import json
import base64
import time
from pathlib import Path
from io import BytesIO
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field, asdict
from datetime import datetime

import numpy as np
import requests
import gradio as gr
from PIL import Image, ImageDraw

print(f"Python: {sys.version}")
print(f"Gradio: {gr.__version__}")

In [ ]:
# Setup Groq API Key
# Option 1: Set via environment variable
# os.environ["GROQ_API_KEY"] = "your-api-key-here"

# Option 2: Get from Colab secrets
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✓ Groq API key loaded from Colab secrets")
except:
    if "GROQ_API_KEY" in os.environ:
        print("✓ Groq API key found in environment")
    else:
        print("⚠ No Groq API key found. RAG features will be limited.")
        print("  Get a free key at: https://console.groq.com/keys")

## 2️⃣ Core Classes

In [ ]:
# =============================================================================
# Data Classes
# =============================================================================

@dataclass
class ChatMessage:
    """Single message in conversation."""
    role: str  # "user", "assistant", "system"
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    image_hash: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ImageSession:
    """Session state for a single image."""
    image_hash: str
    image_b64: str
    triage_result: Optional[Dict] = None
    detection_results: List[Dict] = field(default_factory=list)
    verified_boxes: List[List[float]] = field(default_factory=list)
    masks: List[Any] = field(default_factory=list)

@dataclass
class ConversationState:
    """Full conversation state with memory."""
    session_id: str
    messages: List[ChatMessage] = field(default_factory=list)
    images: Dict[str, ImageSession] = field(default_factory=dict)
    current_image_hash: Optional[str] = None
    summary: str = ""
    turn_count: int = 0
    
    def add_message(self, role: str, content: str, image_hash: str = None, **metadata):
        msg = ChatMessage(
            role=role,
            content=content,
            image_hash=image_hash or self.current_image_hash,
            metadata=metadata
        )
        self.messages.append(msg)
        if role == "user":
            self.turn_count += 1
    
    def get_recent_context(self, n_turns: int = 3) -> str:
        recent = self.messages[-n_turns * 2:] if len(self.messages) > n_turns * 2 else self.messages
        lines = []
        for msg in recent:
            prefix = "User" if msg.role == "user" else "Assistant"
            lines.append(f"{prefix}: {msg.content[:300]}")
        return "\n".join(lines)
    
    def export_for_gradio(self) -> List[List[str]]:
        history = []
        for i in range(0, len(self.messages) - 1, 2):
            user_msg = self.messages[i] if i < len(self.messages) else None
            asst_msg = self.messages[i + 1] if i + 1 < len(self.messages) else None
            if user_msg and user_msg.role == "user":
                history.append([user_msg.content, asst_msg.content if asst_msg else ""])
        return history

@dataclass
class PipelineResult:
    """Pipeline execution result."""
    triage_modality: str = ""
    triage_confidence: float = 0.0
    llava_response: str = ""
    rag_context: str = ""
    boxes: List[List[float]] = field(default_factory=list)
    masks: List[Any] = field(default_factory=list)
    execution_time: float = 0.0
    errors: List[str] = field(default_factory=list)

print("✓ Data classes defined")

In [ ]:
# =============================================================================
# Utility Functions
# =============================================================================

import hashlib

def image_to_base64(image) -> str:
    """Convert image to base64 string."""
    if isinstance(image, str):
        if image.startswith("data:image"):
            return image.split(",", 1)[1]
        if len(image) > 500 and not Path(image).exists():
            return image  # Already base64
        image = Image.open(image)
    elif isinstance(image, Path):
        image = Image.open(image)
    
    if isinstance(image, Image.Image):
        buffered = BytesIO()
        img_format = "PNG" if image.mode == "RGBA" else "JPEG"
        image.save(buffered, format=img_format)
        return base64.b64encode(buffered.getvalue()).decode("utf-8")
    
    raise ValueError(f"Unsupported image type: {type(image)}")

def base64_to_image(b64_string: str) -> Image.Image:
    """Convert base64 string to PIL Image."""
    if b64_string.startswith("data:image"):
        b64_string = b64_string.split(",", 1)[1]
    img_bytes = base64.b64decode(b64_string)
    return Image.open(BytesIO(img_bytes))

def compute_image_hash(image) -> str:
    """Compute hash for image identification."""
    if isinstance(image, Image.Image):
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_bytes = buffered.getvalue()
    elif isinstance(image, str):
        img_bytes = base64.b64decode(image) if len(image) > 500 else open(image, 'rb').read()
    else:
        img_bytes = image
    
    return hashlib.md5(img_bytes).hexdigest()[:12]

def crop_image_by_box(image, box: List[float], padding: int = 5) -> Image.Image:
    """Crop image region by bounding box."""
    if isinstance(image, str):
        image = base64_to_image(image)
    
    x1, y1, x2, y2 = map(int, box)
    w, h = image.size
    x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
    x2, y2 = min(w, x2 + padding), min(h, y2 + padding)
    
    return image.crop((x1, y1, x2, y2))

def draw_boxes_on_image(image, boxes: List[List[float]], labels: List[str] = None) -> Image.Image:
    """Draw bounding boxes on image."""
    if isinstance(image, str):
        image = base64_to_image(image)
    
    image = image.copy()
    draw = ImageDraw.Draw(image)
    
    colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange']
    
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        color = colors[i % len(colors)]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        
        if labels and i < len(labels):
            draw.text((x1, y1 - 15), labels[i], fill=color)
    
    return image

print("✓ Utility functions defined")

In [ ]:
# =============================================================================
# Medical RAG Engine with Groq
# =============================================================================

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    print("⚠ Groq not installed. RAG will use fallback.")

class MedicalRAG:
    """Medical RAG using Groq API."""
    
    def __init__(self, api_key: str = None):
        self.api_key = api_key or os.getenv("GROQ_API_KEY")
        self.client = None
        self.model = "llama3-70b-8192"
        
        if GROQ_AVAILABLE and self.api_key:
            self.client = Groq(api_key=self.api_key)
            print("✓ Groq client initialized")
        else:
            print("⚠ RAG running in fallback mode")
        
        # Simple in-memory knowledge base
        self.knowledge_base = self._init_medical_kb()
    
    def _init_medical_kb(self) -> List[Dict]:
        """Initialize sample medical knowledge base."""
        return [
            {"topic": "pneumonia", "text": "Pneumonia is an infection of the lungs. Treatment includes antibiotics for bacterial pneumonia. Chest X-rays typically show consolidation or infiltrates."},
            {"topic": "tumor", "text": "Tumors can be benign or malignant. Detection often involves imaging studies like CT, MRI, or ultrasound. Biopsy is needed for definitive diagnosis."},
            {"topic": "fracture", "text": "Fractures are breaks in bones. X-rays are the primary diagnostic tool. Treatment depends on severity - immobilization, casting, or surgery may be needed."},
            {"topic": "covid", "text": "COVID-19 causes ground-glass opacities in CT scans. Common symptoms include fever, cough, and respiratory distress. Treatment is supportive with antivirals in some cases."},
            {"topic": "nodule", "text": "Pulmonary nodules are small growths in the lungs. Most are benign but some may indicate cancer. Follow-up imaging or biopsy may be needed based on size and characteristics."},
        ]
    
    def retrieve(self, query: str, top_k: int = 2) -> List[str]:
        """Simple keyword-based retrieval."""
        query_lower = query.lower()
        results = []
        
        for doc in self.knowledge_base:
            if doc["topic"] in query_lower or any(word in query_lower for word in doc["text"].lower().split()[:5]):
                results.append(doc["text"])
        
        return results[:top_k]
    
    def generate(self, prompt: str, system_prompt: str = None) -> str:
        """Generate response using Groq."""
        if not self.client:
            return "[RAG unavailable - no API key]"
        
        messages = [
            {"role": "system", "content": system_prompt or "You are a medical AI assistant."},
            {"role": "user", "content": prompt}
        ]
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.3,
                max_tokens=512
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"[RAG error: {str(e)}]"
    
    def query(self, question: str, context: str = "") -> str:
        """Full RAG query."""
        # Retrieve relevant knowledge
        retrieved = self.retrieve(question)
        knowledge = "\n".join(retrieved) if retrieved else "No specific knowledge found."
        
        # Build prompt
        prompt = f"""Based on the following context and medical knowledge, answer the question.

Visual Analysis Context:
{context}

Medical Knowledge:
{knowledge}

Question: {question}

Answer (be concise and accurate):"""
        
        return self.generate(prompt)

# Initialize RAG
rag_engine = MedicalRAG()
print("✓ MedicalRAG initialized")

In [ ]:
# =============================================================================
# TriMedOrchestrator V2 - Multi-turn with RAG
# =============================================================================

# Worker URLs (adjust for your setup)
WORKER_URLS = {
    "biomedclip": "http://localhost:21006/worker_generate",
    "llava": "http://localhost:21002/worker_generate_stream",
    "dino": "http://localhost:21003/worker_generate",
    "medsam": "http://localhost:21004/worker_generate"
}

# Triage labels
TRIAGE_LABELS = [
    "Chest X-ray", "Brain MRI", "Abdominal CT", "Histopathology",
    "Ultrasound", "Dermoscopy", "Bone X-ray", "Lung CT"
]

# Action keywords
ACTION_KEYWORDS = ["find", "detect", "locate", "segment", "where is", "show", "mark"]
RAG_KEYWORDS = ["treatment", "therapy", "prognosis", "cause", "symptom", "diagnose", "protocol"]


class TriMedOrchestratorV2:
    """Enhanced Orchestrator with Multi-turn Memory and RAG."""
    
    def __init__(self, rag_engine: MedicalRAG = None, timeout: int = 60):
        self.worker_urls = WORKER_URLS
        self.rag = rag_engine
        self.timeout = timeout
        self.state: Optional[ConversationState] = None
    
    # =========================================================================
    # Session Management
    # =========================================================================
    
    def start_session(self, session_id: str = None) -> str:
        session_id = session_id or f"session_{int(time.time())}"
        self.state = ConversationState(session_id=session_id)
        return session_id
    
    def clear_session(self):
        if self.state:
            self.state = ConversationState(session_id=self.state.session_id)
    
    def set_image(self, image) -> str:
        if not self.state:
            self.start_session()
        
        image_b64 = image_to_base64(image)
        image_hash = compute_image_hash(image_b64)
        
        if image_hash not in self.state.images:
            self.state.images[image_hash] = ImageSession(
                image_hash=image_hash,
                image_b64=image_b64
            )
        
        self.state.current_image_hash = image_hash
        return image_hash
    
    def get_current_image(self) -> Optional[ImageSession]:
        if self.state and self.state.current_image_hash:
            return self.state.images.get(self.state.current_image_hash)
        return None
    
    # =========================================================================
    # API Wrappers
    # =========================================================================
    
    def _call_api(self, worker: str, payload: Dict, stream: bool = False) -> Dict:
        """Call worker API."""
        url = self.worker_urls.get(worker)
        if not url:
            raise ValueError(f"Unknown worker: {worker}")
        
        try:
            if stream:
                response = requests.post(url, json=payload, stream=True, timeout=self.timeout)
                response.raise_for_status()
                full_text = ""
                for chunk in response.iter_lines(decode_unicode=True):
                    if chunk:
                        try:
                            data = json.loads(chunk.replace("data: ", ""))
                            if "text" in data:
                                full_text = data["text"]
                        except:
                            full_text += chunk
                return {"text": full_text, "success": True}
            else:
                response = requests.post(url, json=payload, timeout=self.timeout)
                response.raise_for_status()
                return response.json()
        except requests.exceptions.ConnectionError:
            raise ConnectionError(f"Worker '{worker}' unreachable at {url}")
        except requests.exceptions.Timeout:
            raise TimeoutError(f"Request to '{worker}' timed out")
    
    def _call_triage(self, image_b64: str) -> Tuple[str, float]:
        """Call BiomedCLIP for triage."""
        try:
            payload = {"image": image_b64, "labels": TRIAGE_LABELS}
            response = self._call_api("biomedclip", payload)
            return response.get("label", "Unknown"), response.get("score", 0.0)
        except Exception as e:
            return f"Error: {e}", 0.0
    
    def _call_llava(self, query: str, image_b64: str, context: str = "") -> str:
        """Call LLaVA for reasoning."""
        # Build prompt with history context
        history_context = self.state.get_recent_context(n_turns=3) if self.state else ""
        
        full_prompt = f"""USER: <image>
{context}

Previous conversation:
{history_context}

Current question: {query}
ASSISTANT:"""
        
        try:
            payload = {
                "prompt": full_prompt,
                "images": [image_b64],
                "temperature": 0.2,
                "max_new_tokens": 512
            }
            response = self._call_api("llava", payload, stream=True)
            return response.get("text", "No response")
        except Exception as e:
            return f"LLaVA error: {e}"
    
    def _call_dino(self, query: str, image_b64: str) -> Tuple[List, List, List]:
        """Call Grounding DINO."""
        try:
            payload = {"image": image_b64, "prompt": query}
            response = self._call_api("dino", payload)
            return (
                response.get("boxes", []),
                response.get("labels", []),
                response.get("scores", [])
            )
        except Exception as e:
            return [], [], []
    
    def _call_gatekeeper(self, image_b64: str, box: List, target: str) -> Tuple[bool, float]:
        """Call BiomedCLIP gatekeeper."""
        try:
            cropped = crop_image_by_box(image_b64, box)
            cropped_b64 = image_to_base64(cropped)
            
            payload = {
                "image": cropped_b64,
                "labels": [f"Pathology of {target}", "Normal tissue"]
            }
            response = self._call_api("biomedclip", payload)
            
            all_scores = response.get("all_scores", {})
            path_score = all_scores.get(f"Pathology of {target}", 0.0)
            
            return path_score > 0.6, path_score
        except:
            return True, 0.5  # Default to accepting on error
    
    def _call_medsam(self, image_b64: str, boxes: List) -> List:
        """Call MedSAM for segmentation."""
        if not boxes:
            return []
        try:
            payload = {"image": image_b64, "boxes": boxes}
            response = self._call_api("medsam", payload)
            return response.get("masks", [])
        except:
            return []
    
    # =========================================================================
    # Helper Methods
    # =========================================================================
    
    def _should_detect(self, query: str) -> bool:
        return any(kw in query.lower() for kw in ACTION_KEYWORDS)
    
    def _should_use_rag(self, query: str, response: str = "") -> bool:
        combined = (query + " " + response).lower()
        return any(kw in combined for kw in RAG_KEYWORDS)
    
    def _extract_target(self, query: str) -> str:
        import re
        patterns = [
            r"find (?:the |a |an )?(\w+)",
            r"detect (?:the |a |an )?(\w+)",
            r"locate (?:the |a |an )?(\w+)",
        ]
        for p in patterns:
            m = re.search(p, query.lower())
            if m:
                return m.group(1)
        return "abnormality"
    
    # =========================================================================
    # Main Chat Interface
    # =========================================================================
    
    def chat(self, user_input: str, image=None) -> Tuple[str, PipelineResult]:
        """
        Main chat interface with multi-turn memory.
        
        Returns:
            Tuple of (response_text, PipelineResult)
        """
        start_time = time.time()
        
        if not self.state:
            self.start_session()
        
        # Handle image
        if image is not None:
            self.set_image(image)
        
        img_session = self.get_current_image()
        if not img_session:
            return "Please upload an image first.", PipelineResult()
        
        result = PipelineResult()
        
        # Add user message
        self.state.add_message("user", user_input)
        
        try:
            # Stage 1: Triage (if not done)
            if not img_session.triage_result:
                modality, confidence = self._call_triage(img_session.image_b64)
                img_session.triage_result = {"modality": modality, "confidence": confidence}
            
            result.triage_modality = img_session.triage_result["modality"]
            result.triage_confidence = img_session.triage_result["confidence"]
            triage_context = f"[Image type: {result.triage_modality} ({result.triage_confidence:.0%} confidence)]"
            
            # Stage 2: LLaVA Reasoning
            llava_response = self._call_llava(user_input, img_session.image_b64, triage_context)
            result.llava_response = llava_response
            
            # Stage 2.5: RAG Enhancement
            if self.rag and self._should_use_rag(user_input, llava_response):
                rag_answer = self.rag.query(user_input, context=llava_response)
                result.rag_context = rag_answer
                result.llava_response += f"\n\n📚 **Medical Reference**:\n{rag_answer}"
            
            # Stage 3-5: Detection & Segmentation
            if self._should_detect(user_input):
                target = self._extract_target(user_input)
                boxes, labels, scores = self._call_dino(target, img_session.image_b64)
                
                # Gatekeeper
                verified = []
                for box in boxes:
                    is_valid, _ = self._call_gatekeeper(img_session.image_b64, box, target)
                    if is_valid:
                        verified.append(box)
                
                result.boxes = verified
                
                # Segmentation
                if verified:
                    result.masks = self._call_medsam(img_session.image_b64, verified)
                
                # Update session
                img_session.verified_boxes = verified
                img_session.masks = result.masks
                
                # Add detection info to response
                det_info = f"\n\n🔍 **Detection**: Found {len(boxes)} regions, {len(verified)} verified"
                if result.masks:
                    det_info += f", {len(result.masks)} masks generated"
                result.llava_response += det_info
        
        except Exception as e:
            result.errors.append(str(e))
            result.llava_response = f"Error: {str(e)}"
        
        result.execution_time = time.time() - start_time
        
        # Add assistant response
        self.state.add_message("assistant", result.llava_response)
        
        return result.llava_response, result
    
    def get_chat_history(self) -> List[List[str]]:
        """Get history in Gradio format."""
        if not self.state:
            return []
        return self.state.export_for_gradio()
    
    def health_check(self) -> Dict[str, bool]:
        """Check worker connectivity."""
        status = {}
        for name, url in self.worker_urls.items():
            try:
                base = url.rsplit("/", 1)[0]
                r = requests.get(base, timeout=3)
                status[name] = r.status_code < 500
            except:
                status[name] = False
        return status

# Create global orchestrator
orchestrator = TriMedOrchestratorV2(rag_engine=rag_engine)
orchestrator.start_session()
print("✓ TriMedOrchestratorV2 initialized")

## 3️⃣ Gradio Chat Interface

In [ ]:
# =============================================================================
# Gradio Chat Interface
# =============================================================================

def process_chat(message: str, history: List, image: Image.Image = None):
    """
    Process chat message with orchestrator.
    """
    global orchestrator
    
    if not message.strip():
        return history, None
    
    # Chat with orchestrator
    response, result = orchestrator.chat(message, image)
    
    # Update history
    history.append([message, response])
    
    # Generate visualization if boxes detected
    output_image = None
    if result.boxes and orchestrator.get_current_image():
        img_session = orchestrator.get_current_image()
        output_image = draw_boxes_on_image(img_session.image_b64, result.boxes)
    
    return history, output_image


def clear_chat():
    """Clear conversation."""
    global orchestrator
    orchestrator.clear_session()
    return [], None, None


def update_image(image):
    """Handle new image upload."""
    global orchestrator
    if image is not None:
        orchestrator.set_image(image)
    return image


# Build Gradio Interface
with gr.Blocks(title="TriMedAgent V2", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏥 TriMedAgent V2 - Multi-turn Medical AI Chat
    
    Upload a medical image and ask questions. The system provides:
    - 🔬 Visual analysis with LLaVA-Med
    - 📚 Medical knowledge retrieval (RAG)
    - 🎯 Object detection with Grounding DINO
    - 🖼️ Segmentation with MedSAM
    - 💬 Multi-turn conversation memory
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Medical Image", type="pil")
            output_image = gr.Image(label="Detection Result", type="pil")
            
            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear", variant="secondary")
        
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=500,
                bubble_full_width=False
            )
            
            with gr.Row():
                msg_input = gr.Textbox(
                    label="Your message",
                    placeholder="Ask about the image, e.g., 'What do you see?' or 'Find the tumor'",
                    scale=4
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
    
    gr.Markdown("""
    ### 💡 Example Questions:
    - "What type of medical image is this?"
    - "Describe any abnormalities you see"
    - "Find the tumor/nodule/fracture"
    - "What is the treatment for this condition?"
    - "Segment the detected region"
    """)
    
    # Event handlers
    input_image.change(
        fn=update_image,
        inputs=[input_image],
        outputs=[input_image]
    )
    
    send_btn.click(
        fn=process_chat,
        inputs=[msg_input, chatbot, input_image],
        outputs=[chatbot, output_image]
    ).then(
        fn=lambda: "",
        outputs=[msg_input]
    )
    
    msg_input.submit(
        fn=process_chat,
        inputs=[msg_input, chatbot, input_image],
        outputs=[chatbot, output_image]
    ).then(
        fn=lambda: "",
        outputs=[msg_input]
    )
    
    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, input_image, output_image]
    )

print("✓ Gradio interface built")

In [ ]:
# Launch the demo
demo.launch(share=True, debug=True)

## 4️⃣ Testing Without Workers (Simulation Mode)

In [ ]:
# =============================================================================
# Simulation Mode for Testing Without Workers
# =============================================================================

class SimulatedOrchestrator(TriMedOrchestratorV2):
    """
    Orchestrator with simulated responses for testing.
    Use this when workers are not running.
    """
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.simulation_mode = True
    
    def _call_triage(self, image_b64: str) -> Tuple[str, float]:
        """Simulated triage."""
        return "Chest X-ray", 0.85
    
    def _call_llava(self, query: str, image_b64: str, context: str = "") -> str:
        """Simulated LLaVA response."""
        responses = {
            "describe": "This appears to be a chest X-ray showing the thoracic region. I can observe the lung fields, cardiac silhouette, and skeletal structures.",
            "find": "I have identified a potential abnormality in the right upper lobe region that requires further investigation.",
            "treatment": "Based on the imaging findings, I recommend consulting with a pulmonologist for further evaluation.",
        }
        
        query_lower = query.lower()
        for key, response in responses.items():
            if key in query_lower:
                return response
        
        return f"[Simulated] Analyzing the image based on your query: {query}"
    
    def _call_dino(self, query: str, image_b64: str) -> Tuple[List, List, List]:
        """Simulated detection."""
        # Return dummy boxes
        return [
            [100, 100, 200, 200],
            [300, 150, 400, 250]
        ], ["abnormality", "abnormality"], [0.75, 0.65]
    
    def _call_gatekeeper(self, image_b64: str, box: List, target: str) -> Tuple[bool, float]:
        """Simulated gatekeeper."""
        return True, 0.7
    
    def _call_medsam(self, image_b64: str, boxes: List) -> List:
        """Simulated segmentation."""
        return [{"mask_id": i, "area": 1000} for i in range(len(boxes))]


# Test with simulation
print("=" * 50)
print("Testing with Simulated Orchestrator")
print("=" * 50)

sim_orchestrator = SimulatedOrchestrator(rag_engine=rag_engine)
sim_orchestrator.start_session("test_sim")

# Create test image
test_img = Image.new('RGB', (512, 512), color='white')

# Test conversation
queries = [
    "What type of image is this?",
    "Describe what you see",
    "Find any abnormalities",
    "What is the treatment for this?"
]

for i, q in enumerate(queries):
    print(f"\n[Turn {i+1}] User: {q}")
    response, result = sim_orchestrator.chat(q, test_img if i == 0 else None)
    print(f"Assistant: {response[:200]}..." if len(response) > 200 else f"Assistant: {response}")
    print(f"  - Triage: {result.triage_modality} ({result.triage_confidence:.0%})")
    print(f"  - Boxes: {len(result.boxes)}, Masks: {len(result.masks)}")
    print(f"  - Time: {result.execution_time:.2f}s")

In [ ]:
# =============================================================================
# Test RAG Separately
# =============================================================================

print("\n" + "=" * 50)
print("Testing Medical RAG")
print("=" * 50)

# Test queries
rag_queries = [
    "What is the treatment for pneumonia?",
    "How do you diagnose a tumor?",
    "What are the symptoms of COVID-19?"
]

for q in rag_queries:
    print(f"\nQuery: {q}")
    
    # Test retrieval
    retrieved = rag_engine.retrieve(q)
    print(f"Retrieved {len(retrieved)} documents")
    
    # Test full query (if API key available)
    if rag_engine.client:
        answer = rag_engine.query(q)
        print(f"Answer: {answer[:200]}..." if len(answer) > 200 else f"Answer: {answer}")
    else:
        print("(Skipping generation - no API key)")

## 📖 Documentation

### Multi-turn Memory

The `TriMedOrchestratorV2` maintains conversation state across turns:

```python
# State is automatically managed
orchestrator.chat("What do you see?", image)  # Turn 1
orchestrator.chat("Is it malignant?")          # Turn 2 (remembers Turn 1)
orchestrator.chat("What's the treatment?")     # Turn 3 (remembers all)
```

### RAG Integration

RAG is automatically triggered when queries contain keywords like:
- `treatment`, `therapy`, `prognosis`
- `cause`, `symptom`, `diagnose`
- `protocol`, `guideline`

### Worker Configuration

Update `WORKER_URLS` to match your setup:

```python
WORKER_URLS = {
    "biomedclip": "http://your-server:21006/worker_generate",
    "llava": "http://your-server:21002/worker_generate_stream",
    "dino": "http://your-server:21003/worker_generate",
    "medsam": "http://your-server:21004/worker_generate"
}
```